In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip -q install xgboost lightgbm shap joblib catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.3 MB/s eta 0:00:00


In [ ]:
# ============================================================
# Standard Stacking Ensemble
# ============================================================

import warnings
import joblib
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
)

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# ============================================================
# Project Directories
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/Biomass_Project")

NESTED_CV_DIR = PROJECT_DIR / "10_Tables" / "Nested_CV"

ENSEMBLE_DIR = PROJECT_DIR / "08_Ensemble"

MODEL_DIR = ENSEMBLE_DIR / "Models"
PRED_DIR = ENSEMBLE_DIR / "Predictions"
TABLE_DIR = ENSEMBLE_DIR / "Tables"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ============================================================
# Evaluation Function
# ============================================================

def evaluate_model(y_true, y_pred):

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    return {

        "R2": r2_score(y_true, y_pred),

        "RMSE": rmse,

        "MAE": mean_absolute_error(y_true, y_pred),

        "MAPE": np.mean(
            np.abs((y_true-y_pred)/y_true)
        )*100,

        "rRMSE": rmse/np.mean(y_true)*100

    }

In [ ]:
# ============================================================
# Load Nested CV OOF Predictions
# ============================================================

model_names = ["RF", "XGB", "CatBoost", "LGBM", "SVR"]

oof_predictions = {}
reference_df = None

for model in model_names:

    file_path = NESTED_CV_DIR / f"{model}_Predictions.csv"

    df = pd.read_csv(file_path)

    required_cols = ["Index", "Observed", "Predicted"]

    if not all(col in df.columns for col in required_cols):
        raise ValueError(f"{model} prediction file is missing required columns.")

    if reference_df is None:
        reference_df = df[["Index", "Observed"]].copy()
    else:
        if not reference_df["Index"].equals(df["Index"]):
            raise ValueError(f"Index mismatch detected in {model}.")

    oof_predictions[model] = df

print("✓ All prediction files loaded successfully.")

✓ All prediction files loaded successfully.


In [ ]:
# ============================================================
# Create Meta-Feature Matrix
# ============================================================

X_meta = np.column_stack([

    oof_predictions["RF"]["Predicted"].values,
    oof_predictions["XGB"]["Predicted"].values,
    oof_predictions["CatBoost"]["Predicted"].values,
    oof_predictions["LGBM"]["Predicted"].values,
    oof_predictions["SVR"]["Predicted"].values,

])

feature_names = [
    "RF_Pred",
    "XGB_Pred",
    "CatBoost_Pred",
    "LGBM_Pred",
    "SVR_Pred"
]

y = reference_df["Observed"].values

print(f"Meta-feature matrix shape : {X_meta.shape}")
print(f"Target shape              : {y.shape}")

Meta-feature matrix shape : (1227, 5)
Target shape              : (1227,)


In [ ]:
# ============================================================
# Meta-Learners
# ============================================================

meta_models = {

    "RF": RandomForestRegressor(

        n_estimators=650,
        max_depth=9,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features=None,
        random_state=RANDOM_STATE,
        n_jobs=-1

    ),

    "XGB": XGBRegressor(

        n_estimators=468,
        max_depth=10,
        learning_rate=0.01,
        subsample=0.6,
        colsample_bytree=1.0,
        reg_alpha=0,
        reg_lambda=1,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1

    ),

    "CatBoost": CatBoostRegressor(

        iterations=316,
        depth=5,
        learning_rate=0.03,
        l2_leaf_reg=4,
        random_strength=1,
        subsample=0.8,
        random_state=RANDOM_STATE,
        verbose=0

    )

}

In [ ]:
# ============================================================
# Train Meta-Learners
# ============================================================

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

results = []

for model_name, model in meta_models.items():

    print(f"\nTraining {model_name} meta-learner...")

    oof_pred = np.zeros(len(y))

    for train_idx, test_idx in kf.split(X_meta):

        X_train = X_meta[train_idx]
        X_test = X_meta[test_idx]

        y_train = y[train_idx]

        model.fit(X_train, y_train)

        oof_pred[test_idx] = model.predict(X_test)

    metrics = evaluate_model(y, oof_pred)

    results.append({
        "Model": model_name,
        "Predictions": oof_pred,
        "Metrics": metrics
    })

print("\n✓ Training completed.")


Training RF meta-learner...

Training XGB meta-learner...

Training CatBoost meta-learner...

✓ Training completed.


In [ ]:
# ============================================================
# Train Final Models
# ============================================================

final_models = {}

for model_name, model in meta_models.items():

    model.fit(X_meta, y)

    final_models[model_name] = model

print("✓ Final models trained.")

# Save feature names for SHAP
feature_names = [
    "RF_Pred",
    "XGB_Pred",
    "CatBoost_Pred",
    "LGBM_Pred",
    "SVR_Pred"
]

joblib.dump(
    feature_names,
    MODEL_DIR / "Stacking_Feature_Names.pkl"
)

print("✓ Feature names saved.")

✓ Final models trained.
✓ Feature names saved.


In [ ]:
# ============================================================
# Save Models
# ============================================================

for model_name, model in final_models.items():

    joblib.dump(
        model,
        MODEL_DIR / f"{model_name}_Stacking_Model.pkl"
    )

print("✓ Models saved.")

✓ Models saved.


In [ ]:
# ============================================================
# Save OOF Predictions
# ============================================================

for result in results:

    pred_df = pd.DataFrame({

        "Index": reference_df["Index"],
        "Observed": y,
        "Predicted": result["Predictions"]

    })

    pred_df.to_csv(
        PRED_DIR / f"{result['Model']}_Stacking_Prediction.csv",
        index=False
    )

print("✓ Prediction files saved.")

✓ Prediction files saved.


In [ ]:
# ============================================================
# Performance Summary
# ============================================================

summary = []

for result in results:

    row = {

        "Model": result["Model"],
        **result["Metrics"]

    }

    summary.append(row)

summary_df = (
    pd.DataFrame(summary)
    .sort_values("R2", ascending=False)
    .reset_index(drop=True)
)

display(summary_df)

summary_df.to_csv(
    TABLE_DIR / "Stacking_Performance.csv",
    index=False
)

print("✓ Performance table saved.")

,Model,R2,RMSE,MAE,MAPE,rRMSE
0,CatBoost,0.733394,32.456132,19.772834,18.615061,27.693652
1,RF,0.716755,33.453610,20.264056,18.343411,28.544764
2,XGB,0.699228,34.473075,21.073863,19.159797,29.414637


✓ Performance table saved.


In [ ]:
from sklearn.linear_model import Ridge

In [ ]:
meta_models = {
    "Ridge": Ridge(
        alpha=1.0,
        random_state=RANDOM_STATE
    )
}

In [ ]:
# ============================================================
# Train Meta-Learners
# ============================================================

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

results = []

for model_name, model in meta_models.items():

    print(f"\nTraining {model_name} meta-learner...")

    oof_pred = np.zeros(len(y))

    for train_idx, test_idx in kf.split(X_meta):

        X_train = X_meta[train_idx]
        X_test = X_meta[test_idx]

        y_train = y[train_idx]

        model.fit(X_train, y_train)

        oof_pred[test_idx] = model.predict(X_test)

    metrics = evaluate_model(y, oof_pred)

    results.append({
        "Model": model_name,
        "Predictions": oof_pred,
        "Metrics": metrics
    })

print("\n✓ Training completed.")


Training Ridge meta-learner...

✓ Training completed.


In [ ]:
# ============================================================
# Train Final Models
# ============================================================

final_models = {}

for model_name, model in meta_models.items():

    model.fit(X_meta, y)

    final_models[model_name] = model

print("✓ Final models trained.")

✓ Final models trained.


In [ ]:
# ============================================================
# Save Models
# ============================================================

for model_name, model in final_models.items():

    joblib.dump(
        model,
        MODEL_DIR / f"{model_name}_Stacking_Model.pkl"
    )

print("✓ Models saved.")

✓ Models saved.


In [ ]:
# ============================================================
# Save OOF Predictions
# ============================================================

for result in results:

    pred_df = pd.DataFrame({

        "Index": reference_df["Index"],
        "Observed": y,
        "Predicted": result["Predictions"]

    })

    pred_df.to_csv(
        PRED_DIR / f"{result['Model']}_Stacking_Prediction.csv",
        index=False
    )

print("✓ Prediction files saved.")

✓ Prediction files saved.


In [ ]:
# ============================================================
# Performance Summary
# ============================================================

summary = []

for result in results:

    row = {

        "Model": result["Model"],
        **result["Metrics"]

    }

    summary.append(row)

summary_df = (
    pd.DataFrame(summary)
    .sort_values("R2", ascending=False)
    .reset_index(drop=True)
)

display(summary_df)

summary_df.to_csv(
    TABLE_DIR / "Stacking_Performance.csv",
    index=False
)

print("✓ Performance table saved.")

,Model,R2,RMSE,MAE,MAPE,rRMSE
0,Ridge,0.736117,32.289962,19.547547,18.288102,27.551865


✓ Performance table saved.
